<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/M0_AUDIT_v1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M0_AUDIT_v1.1 — DeepBind FEMTO M0 감사 종료 노트북

---

| 항목 | 내용 |
|:---|:---|
| **버전** | M0_AUDIT_v1.1 |
| **목적** | M0 감사 루프 종료 및 M0-BRIDGE-2048 진행 여부 판정 |
| **기준 모델** | `M0_FEMTO_Baseline_v1_baseline고정.ipynb` (동결 기준) |
| **M0 재구현** | **금지** — 기준 노트북 로직을 추정하여 재구현하지 않음 |
| **v1 재계산 결과** | `REFERENCE_REIMPLEMENTATION_REJECTED` — 기준으로 사용하지 않음 |

---

## 핵심 판정 원칙

1. 기존 v1의 `m0_learning_recomputed.csv`는 기준 M0 재현 결과로 사용하지 않는다.
2. `K=6.0`은 Mahalanobis 거리의 고정 임계값이라고 임의 해석하지 않는다.
3. 기준 노트북의 실제 로직을 완전히 추출하지 못하면 재구현하지 않는다.
4. 기준 노트북 자동 재실행 실패만으로 전체 감사를 실패시키지 않는다.
5. `BASELINE_M0_frozen.json`과 `m0_baseline_result.csv`를 동결 기준 산출물로 사용한다.
6. `frozen JSON`의 `pass_criteria.max_reg_FAR=0.0`과 실제 `max_reg_FAR=0.0088`의 충돌은 `CRITERIA_WARNING`으로 보고하며 frozen JSON은 수정하지 않는다.
7. 다음 조건이 충족되면 최종 상태를 `AUDIT_PASS_WITH_CRITERIA_WARNING`으로 한다:
   - 데이터 구조 6/11/11/28 일치
   - Learning 6개 레코드 식별 성공
   - legacy CSV와 frozen JSON 핵심 통계가 표시 정밀도 범위에서 일치
   - 기준 노트북 발견 및 SHA-256 기록
   - 원본 파일의 감사 전후 SHA-256 동일
8. 위 조건을 만족하면 `recommended_next_action = READY_FOR_M0_BRIDGE_2048`
9. 기존 v1 재계산 불일치는 `REFERENCE_REIMPLEMENTATION_REJECTED`로 기록하고 전체 감사 실패 사유로 사용하지 않는다.

## Read-only 원칙

쓰기 작업은 `audit_outputs/YYYYMMDD_HHMMSS_v1_1/` 폴더 안에서만 허용된다.

절대 수정 금지 파일:
- `BASELINE_M0_frozen.json`
- `m0_baseline_result.csv`
- `M0_FEMTO_Baseline_v1.ipynb`
- `M0_FEMTO_Baseline_v1_baseline.ipynb`
- `M0_FEMTO_Baseline_v1_baseline고정.ipynb`
- `OnlineSim_FEMTO_v3.ipynb`
- FEMTO 원본 CSV 전체
- 기존 `audit_outputs` 산출물

## 최종 보고서 해석 규칙

1. `final_status = AUDIT_PASS_WITH_CRITERIA_WARNING` → M0 감사 종료, 다음 단계 진행
2. `FROZEN_BASELINE_SELF_CRITERIA_CONTRADICTION`은 예상된 경고 — `AUDIT_HOLD`/`AUDIT_FAILED`로 변경하지 않음
3. v1 재계산 불일치 = 원본 M0 실패가 아닌 감사 재구현 실패
4. M0 알고리즘을 새로 구현하거나 `threshold=6.0`으로 다시 계산하지 않음
5. `READY_FOR_M0_BRIDGE_2048`이 나오면 추가 M0 감사 노트북을 만들지 않고 다음 단계로 이동

In [1]:
# ================================================================
# DeepBind M0_AUDIT_v1.1
# 목적:
# 1) M0 감사 루프 종료
# 2) 동결 기준 산출물과 데이터 구조 검증
# 3) 잘못된 v1 재구현 결과를 기준 결과에서 제외
# 4) M0-BRIDGE-2048 진행 여부 판정
#
# 중요:
# - 원본 파일 수정 금지
# - M0 알고리즘 임의 재구현 금지
# - K=6.0을 거리 임계값으로 임의 해석 금지
# ================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import ast
import csv
import hashlib
import json
import math
import os
import re
import shutil
import sys
import traceback

from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd


# ================================================================
# 0. 고정 설정
# ================================================================

AUDIT_VERSION = "M0_AUDIT_v1.1"

EXPECTED_SPLIT_COUNTS = {
    "LEARNING": 6,
    "TEST": 11,
    "FULL_TEST": 11,
    "TOTAL": 28,
}

LEARNING_BEARINGS = [
    "Bearing1_1",
    "Bearing1_2",
    "Bearing2_1",
    "Bearing2_2",
    "Bearing3_1",
    "Bearing3_2",
]

SPLIT_DIR_NAMES = {
    "LEARNING": "Training(Learning)_set",
    "TEST": "Test(Test)_set",
    "FULL_TEST": "Validation(Full_Test)_Set",
}

REFERENCE_NOTEBOOK_CANDIDATES = [
    "M0_FEMTO_Baseline_v1_baseline.ipynb",
    "M0_FEMTO_Baseline_v1_baseline\uace0\uc815.ipynb",
    "M0_FEMTO_Baseline_v1.ipynb",
]

FROZEN_JSON_NAME = "BASELINE_M0_frozen.json"
LEGACY_RESULT_NAME = "m0_baseline_result.csv"

FEMTO_FS = 25600
FEMTO_DECISION_INTERVAL_SEC = 10.0
FEMTO_SNAPSHOT_DURATION_SEC = 0.1
COL_H = 4
K = 6.0
CONSEC = 5
REG_MIN = 90
REG_MAX = 600

# frozen JSON에 저장된 표시 자릿수를 고려한 비교 허용범위
METRIC_TOLERANCES = {
    "n_total_learn": 0.0,
    "n_detected": 0.0,
    "mean_lead_hours": 0.015,
    "max_lead_hours": 0.055,
    "min_lead_hours": 0.055,
    "mean_reg_FAR": 0.00055,
    "max_reg_FAR": 0.00055,
}

PROJECT_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb"),
    Path("/content/drive/My Drive/Colab Notebooks/field_iis3dwb"),
]

FEMTO_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/RunToFailure_Raw_Data/"
         "PRONOSTIA_FEMTO_Bearing"),
    Path("/content/drive/My Drive/Colab Notebooks/RunToFailure_Raw_Data/"
         "PRONOSTIA_FEMTO_Bearing"),
    Path("/content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/"
         "PRONOSTIA_FEMTO_Bearing"),
    Path("/content/drive/My Drive/Colab Notebooks/3.RunToFailure_Raw_Data/"
         "PRONOSTIA_FEMTO_Bearing"),
]


# ================================================================
# 1. 공통 함수
# ================================================================

def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, set):
        return sorted(value)
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return str(value)


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
            default=json_default,
        )


def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def file_metadata(path):
    path = Path(path)
    stat = path.stat()
    return {
        "absolute_path": str(path.resolve()),
        "file_name": path.name,
        "file_size": int(stat.st_size),
        "modified_time": datetime.fromtimestamp(
            stat.st_mtime
        ).isoformat(timespec="seconds"),
        "sha256": sha256_file(path),
    }


def natural_key(value):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", str(value))
    ]


def normalize_column_name(value):
    return re.sub(r"[^a-z0-9\uac00-\ud7a3]+", "", str(value).strip().lower())


def first_existing(candidates):
    for path in candidates:
        if Path(path).exists():
            return Path(path)
    return None


def read_csv_flexible(path):
    path = Path(path)
    encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr", "latin-1"]
    last_error = None
    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as exc:
            last_error = exc
    raise RuntimeError(
        f"CSV를 읽지 못했습니다: {path}\n마지막 오류: {last_error}"
    )


def pick_column(df, aliases, required=True):
    normalized = {
        normalize_column_name(column): column
        for column in df.columns
    }
    # 완전 일치 우선
    for alias in aliases:
        key = normalize_column_name(alias)
        if key in normalized:
            return normalized[key]
    # 부분 문자열 일치
    for alias in aliases:
        key = normalize_column_name(alias)
        for normalized_name, original in normalized.items():
            if key and key in normalized_name:
                return original
    if required:
        raise KeyError(
            f"필수 열을 찾지 못했습니다. aliases={aliases}, "
            f"actual_columns={list(df.columns)}"
        )
    return None


def numeric_series(series):
    cleaned = (
        series.astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({
            "": np.nan, "None": np.nan, "nan": np.nan,
            "NaN": np.nan, "-": np.nan,
        })
    )
    return pd.to_numeric(cleaned, errors="coerce")


def safe_float(value):
    try:
        result = float(value)
        if math.isnan(result):
            return None
        return result
    except Exception:
        return None


def unique_existing_paths(paths):
    seen = set()
    result = []
    for path in paths:
        path = Path(path)
        if not path.exists():
            continue
        resolved = str(path.resolve())
        if resolved not in seen:
            seen.add(resolved)
            result.append(path)
    return result


def locate_project_file(project_root, names, required=True):
    if isinstance(names, str):
        names = [names]
    # 프로젝트 루트 직속 파일 우선
    direct = [
        project_root / name
        for name in names
        if (project_root / name).exists()
    ]
    if direct:
        return direct[0], direct
    # 하위 폴더 검색
    found = []
    for name in names:
        found.extend(project_root.rglob(name))
    found = unique_existing_paths(found)
    found.sort(key=lambda p: (len(p.parts), natural_key(str(p))))
    if found:
        return found[0], found
    if required:
        raise FileNotFoundError(
            f"파일을 찾지 못했습니다: names={names}, root={project_root}"
        )
    return None, []


def list_bearing_dirs(split_dir):
    if split_dir is None or not split_dir.exists():
        return []
    return sorted(
        [
            p for p in split_dir.iterdir()
            if p.is_dir() and re.fullmatch(r"Bearing\d+_\d+", p.name)
        ],
        key=lambda p: natural_key(p.name),
    )


def count_data_files(bearing_dir):
    allowed_suffixes = {".csv", ".txt", ".mat", ".npy", ".npz", ".dat"}
    files = [
        p for p in bearing_dir.iterdir()
        if p.is_file() and p.suffix.lower() in allowed_suffixes
    ]
    return len(files)


print("[STEP 0-1] 공통 함수 정의 완료 ✅")
# assert 기본값 검증
assert CONSEC == 5
assert FEMTO_DECISION_INTERVAL_SEC == 10.0
assert FEMTO_SNAPSHOT_DURATION_SEC == 0.1
assert COL_H == 4
assert K == 6.0
assert REG_MIN == 90
assert REG_MAX == 600
print("[STEP 0-1] 모든 assert 통과 ✅")

Mounted at /content/drive
[STEP 0-1] 공통 함수 정의 완료 ✅
[STEP 0-1] 모든 assert 통과 ✅


In [2]:
# ================================================================
# 2. 프로젝트 및 출력 폴더 결정
# ================================================================

PROJECT_ROOT = first_existing(PROJECT_ROOT_CANDIDATES)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "PROJECT_ROOT를 찾지 못했습니다.\n"
        + "\n".join(str(p) for p in PROJECT_ROOT_CANDIDATES)
    )

FEMTO_ROOT = first_existing(FEMTO_ROOT_CANDIDATES)
if FEMTO_ROOT is None:
    raise FileNotFoundError(
        "FEMTO_ROOT를 찾지 못했습니다.\n"
        + "\n".join(str(p) for p in FEMTO_ROOT_CANDIDATES)
    )

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_v1_1"
OUTPUT_ROOT = PROJECT_ROOT / "audit_outputs" / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)

print("=" * 72)
print(f"AUDIT_VERSION  : {AUDIT_VERSION}")
print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"FEMTO_ROOT     : {FEMTO_ROOT}")
print(f"OUTPUT_ROOT    : {OUTPUT_ROOT}")
print("=" * 72)


# ================================================================
# 3. 기준 파일 탐색
# ================================================================

frozen_json_path, frozen_candidates = locate_project_file(
    PROJECT_ROOT, FROZEN_JSON_NAME, required=True,
)

legacy_csv_path, legacy_candidates = locate_project_file(
    PROJECT_ROOT, LEGACY_RESULT_NAME, required=True,
)

reference_notebook_path, reference_notebook_found = locate_project_file(
    PROJECT_ROOT, REFERENCE_NOTEBOOK_CANDIDATES, required=True,
)

resolution = {
    "audit_version": AUDIT_VERSION,
    "created_at": now_iso(),
    "project_root": str(PROJECT_ROOT),
    "femto_root": str(FEMTO_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "selected_files": {
        "frozen_json": str(frozen_json_path),
        "legacy_result_csv": str(legacy_csv_path),
        "reference_notebook": str(reference_notebook_path),
    },
    "all_candidates": {
        "frozen_json": [str(p) for p in frozen_candidates],
        "legacy_result_csv": [str(p) for p in legacy_candidates],
        "reference_notebooks": [str(p) for p in reference_notebook_found],
    },
    "reference_notebook_selection_rule":
        "project root direct file first; otherwise shortest matching path",
}

write_json(OUTPUT_ROOT / "project_resolution_v1_1.json", resolution)

print(f"[STEP 3] frozen_json        : {frozen_json_path}")
print(f"[STEP 3] legacy_result_csv  : {legacy_csv_path}")
print(f"[STEP 3] reference_notebook : {reference_notebook_path}")
print(f"[STEP 3] project_resolution_v1_1.json 저장 완료")

AUDIT_VERSION  : M0_AUDIT_v1.1
PROJECT_ROOT   : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb
FEMTO_ROOT     : /content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing
OUTPUT_ROOT    : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_055925_v1_1
[STEP 3] frozen_json        : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/BASELINE_M0_frozen.json
[STEP 3] legacy_result_csv  : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/m0_baseline_result.csv
[STEP 3] reference_notebook : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/M0_FEMTO_Baseline_v1_baseline고정.ipynb
[STEP 3] project_resolution_v1_1.json 저장 완료


In [3]:
# ================================================================
# 4. 감사 전 원본 SHA-256
# ================================================================

source_paths = unique_existing_paths([
    frozen_json_path,
    legacy_csv_path,
    reference_notebook_path,
    PROJECT_ROOT / "M0_FEMTO_Baseline_v1.ipynb",
    PROJECT_ROOT / "M0_FEMTO_Baseline_v1_baseline.ipynb",
    PROJECT_ROOT / "M0_FEMTO_Baseline_v1_baseline\uace0\uc815.ipynb",
    PROJECT_ROOT / "OnlineSim_FEMTO_v3.ipynb",
])

source_before = {
    str(path.resolve()): file_metadata(path)
    for path in source_paths
}

print("\n[CHECKPOINT] 원본 파일 감사 전 SHA-256 기록 완료")
for path, metadata in source_before.items():
    print(f"  {metadata['file_name']}: {metadata['sha256'][:16]}...")


[CHECKPOINT] 원본 파일 감사 전 SHA-256 기록 완료
  BASELINE_M0_frozen.json: a4632ad1b9c51976...
  m0_baseline_result.csv: 63f493e6e3482157...
  M0_FEMTO_Baseline_v1_baseline고정.ipynb: 967c36a127d63eee...
  M0_FEMTO_Baseline_v1.ipynb: 921f1bbea7b460d7...
  OnlineSim_FEMTO_v3.ipynb: ce48b07d70af97aa...


In [4]:
# ================================================================
# 5. 기준 노트북 소스 계약 기록
#    - 코드 실행하지 않음
#    - 코드 셀의 해시와 핵심 토큰만 기록
# ================================================================

with reference_notebook_path.open("r", encoding="utf-8") as f:
    reference_notebook = json.load(f)

code_cells = []
all_code = []

for index, cell in enumerate(reference_notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)
    source = str(source)
    all_code.append(source)
    code_cells.append({
        "cell_index": index,
        "source_sha256": hashlib.sha256(source.encode("utf-8")).hexdigest(),
        "source_length": len(source),
        "contains_extract_features": "extract_features" in source,
        "contains_mahal": bool(re.search(r"\bmahal", source, re.IGNORECASE)),
        "contains_consecutive_alarm": "consecutive_alarm" in source,
        "contains_run_one": "run_one" in source,
        "contains_reg_far": bool(re.search(r"reg[_ ]?FAR", source, re.IGNORECASE)),
        "contains_frozen_write":
            FROZEN_JSON_NAME in source and
            bool(re.search(r"json\.dump|open\s*\(", source)),
    })

combined_code = "\n\n".join(all_code)

constant_patterns = {
    "FEMTO_FS": r"\bFEMTO_FS\s*=\s*([0-9.]+)",
    "COL_H": r"\bCOL_H\s*=\s*([0-9.]+)",
    "K": r"(?m)^\s*K(?:_DEFAULT)?\s*=\s*([0-9.]+)",
    "CONSEC": r"\bCONSEC\s*=\s*([0-9.]+)",
    "REG_MIN": r"\b(?:REG_MIN|MIN_WIN)\s*=\s*([0-9.]+)",
    "REG_MAX": r"\b(?:REG_MAX|MAX_WIN)\s*=\s*([0-9.]+)",
}

extracted_constants = {}
for name, pattern in constant_patterns.items():
    match = re.search(pattern, combined_code)
    extracted_constants[name] = (
        safe_float(match.group(1)) if match else None
    )

source_code_contract = {
    "audit_version": AUDIT_VERSION,
    "reference_notebook": file_metadata(reference_notebook_path),
    "code_cell_count": len(code_cells),
    "code_cells": code_cells,
    "extracted_constants": extracted_constants,
    "expected_contract": {
        "FEMTO_FS": FEMTO_FS,
        "COL_H": COL_H,
        "K": K,
        "CONSEC": CONSEC,
        "REG_MIN": REG_MIN,
        "REG_MAX": REG_MAX,
    },
    "execution_policy": "READ_ONLY_STATIC_INSPECTION",
    "algorithm_reimplementation_performed": False,
    "algorithm_execution_performed": False,
    "reason":
        "M0_AUDIT_v1의 임의 재구현 오류를 반복하지 않기 위해 "
        "기준 노트북 코드를 추정하거나 부분 실행하지 않음",
}

write_json(OUTPUT_ROOT / "source_code_contract_v1_1.json", source_code_contract)

notebook_checked = (
    reference_notebook_path.exists()
    and len(code_cells) > 0
    and source_code_contract["reference_notebook"]["sha256"]
)

print(f"[STEP 5] 기준 노트북 코드 셀 수: {len(code_cells)}")
print(f"[STEP 5] 추출 상수: {extracted_constants}")
print(f"[STEP 5] notebook_checked = {bool(notebook_checked)}")
print(f"[STEP 5] source_code_contract_v1_1.json 저장 완료")

[STEP 5] 기준 노트북 코드 셀 수: 6
[STEP 5] 추출 상수: {'FEMTO_FS': 25600.0, 'COL_H': 4.0, 'K': 6.0, 'CONSEC': 5.0, 'REG_MIN': 90.0, 'REG_MAX': 600.0}
[STEP 5] notebook_checked = True
[STEP 5] source_code_contract_v1_1.json 저장 완료


In [5]:
# ================================================================
# 6. 데이터셋 구조 재확인
# ================================================================

dataset_rows = []
split_count_summary = {}

for split_name, split_dir_name in SPLIT_DIR_NAMES.items():
    split_dir = FEMTO_ROOT / split_dir_name

    if not split_dir.exists():
        split_count_summary[split_name] = 0
        dataset_rows.append({
            "split": split_name,
            "record_uid": None,
            "logical_bearing_id": None,
            "bearing_directory": None,
            "file_count": 0,
            "estimated_observation_hours": None,
            "status": "SPLIT_DIRECTORY_NOT_FOUND",
        })
        continue

    bearing_dirs = list_bearing_dirs(split_dir)
    split_count_summary[split_name] = len(bearing_dirs)

    for bearing_dir in bearing_dirs:
        file_count = count_data_files(bearing_dir)
        estimated_hours = (
            ((file_count - 1) * FEMTO_DECISION_INTERVAL_SEC) / 3600.0
            if file_count > 0 else 0.0
        )
        dataset_rows.append({
            "split": split_name,
            "record_uid": f"{split_name}/{bearing_dir.name}",
            "logical_bearing_id": bearing_dir.name,
            "bearing_directory": str(bearing_dir),
            "file_count": file_count,
            "estimated_observation_hours": round(estimated_hours, 4),
            "status": "OK",
        })

dataset_df = pd.DataFrame(dataset_rows)
dataset_df.to_csv(
    OUTPUT_ROOT / "dataset_structure_v1_1.csv",
    index=False,
    encoding="utf-8-sig",
)

split_count_summary["TOTAL"] = sum(
    split_count_summary.get(name, 0)
    for name in ["LEARNING", "TEST", "FULL_TEST"]
)

dataset_count_checks = {
    name: split_count_summary.get(name) == expected
    for name, expected in EXPECTED_SPLIT_COUNTS.items()
}

dataset_structure_pass = all(dataset_count_checks.values())

print("\n[CHECKPOINT] 데이터 구조")
for k, v in split_count_summary.items():
    expected = EXPECTED_SPLIT_COUNTS.get(k, "?")
    ok = "✅" if v == expected else "❌"
    print(f"  {k}: {v} / 기대={expected}  {ok}")
print(f"  dataset_structure_pass = {dataset_structure_pass}")
display(dataset_df[["split","logical_bearing_id","file_count","estimated_observation_hours","status"]])


[CHECKPOINT] 데이터 구조
  LEARNING: 6 / 기대=6  ✅
  TEST: 11 / 기대=11  ✅
  FULL_TEST: 11 / 기대=11  ✅
  TOTAL: 28 / 기대=28  ✅
  dataset_structure_pass = True


,split,logical_bearing_id,file_count,estimated_observation_hours,status
0,LEARNING,Bearing1_1,3269,9.0778,OK
1,LEARNING,Bearing1_2,1015,2.8167,OK
2,LEARNING,Bearing2_1,1062,2.9472,OK
3,LEARNING,Bearing2_2,797,2.2111,OK
4,LEARNING,Bearing3_1,604,1.6750,OK
5,LEARNING,Bearing3_2,1637,4.5444,OK
6,TEST,Bearing1_3,590,1.6361,OK
7,TEST,Bearing1_4,1327,3.6833,OK
8,TEST,Bearing1_5,2677,7.4333,OK
9,TEST,Bearing1_6,2232,6.1972,OK


In [6]:
# ================================================================
# 7. Frozen JSON 및 Legacy CSV 로드 + Learning 레코드 식별
# ================================================================

with frozen_json_path.open("r", encoding="utf-8-sig") as f:
    frozen = json.load(f)

legacy_df = read_csv_flexible(legacy_csv_path)

print("Legacy CSV columns:")
print(list(legacy_df.columns))

bearing_col = pick_column(
    legacy_df,
    ["record_uid", "logical_bearing_id", "bearing", "bearing_id", "베어링"],
)

n_col = pick_column(
    legacy_df,
    ["N", "n_samples", "file_count", "count"],
    required=False,
)

alarm_col = pick_column(
    legacy_df,
    ["alarm_idx", "alarm_start_idx", "legacy_alarm_start_idx",
     "detection_idx", "detect_idx", "alarm"],
    required=False,
)

lead_col = pick_column(
    legacy_df,
    ["lead_hours", "lead_hour", "lead_h", "leadtime_hours",
     "lead_time_hours", "lead"],
)

far_col = pick_column(
    legacy_df,
    ["reg_FAR", "reg_far", "registration_far", "reg_window_exceedance_rate"],
)

legacy_work = legacy_df.copy()
legacy_work["_logical_bearing_id"] = (
    legacy_work[bearing_col]
    .astype(str)
    .str.extract(r"(Bearing\d+_\d+)", expand=False)
)

learning_legacy = legacy_work[
    legacy_work["_logical_bearing_id"].isin(LEARNING_BEARINGS)
].copy()

learning_occurrences = (
    learning_legacy["_logical_bearing_id"]
    .value_counts(dropna=False)
    .to_dict()
)

learning_rows_unique = all(
    learning_occurrences.get(bearing, 0) == 1
    for bearing in LEARNING_BEARINGS
)

learning_legacy["_lead_hours_numeric"] = numeric_series(learning_legacy[lead_col])
learning_legacy["_reg_far_numeric_raw"] = numeric_series(learning_legacy[far_col])

# FAR가 1보다 크면 백분율 표기로 간주
far_values_raw = learning_legacy["_reg_far_numeric_raw"].dropna()
far_percent_scale_applied = (
    len(far_values_raw) > 0 and far_values_raw.abs().max() > 1.0
)

if far_percent_scale_applied:
    learning_legacy["_reg_far_numeric"] = learning_legacy["_reg_far_numeric_raw"] / 100.0
    print("  [FAR] 백분율 스케일 감지 → /100 변환 적용")
else:
    learning_legacy["_reg_far_numeric"] = learning_legacy["_reg_far_numeric_raw"]

if n_col is not None:
    learning_legacy["_N_numeric"] = numeric_series(learning_legacy[n_col])
else:
    learning_legacy["_N_numeric"] = np.nan

if alarm_col is not None:
    learning_legacy["_alarm_idx_numeric"] = numeric_series(learning_legacy[alarm_col])
    detected_mask = (
        learning_legacy["_alarm_idx_numeric"].notna()
        & (learning_legacy["_alarm_idx_numeric"] >= 0)
    )
else:
    detected_mask = learning_legacy["_lead_hours_numeric"].notna()

learning_legacy["_detected"] = detected_mask

learning_output_columns = [
    "_logical_bearing_id",
    "_N_numeric",
    *([ "_alarm_idx_numeric"] if alarm_col is not None else []),
    "_detected",
    "_lead_hours_numeric",
    "_reg_far_numeric_raw",
    "_reg_far_numeric",
]

learning_legacy[learning_output_columns].sort_values(
    "_logical_bearing_id",
    key=lambda s: s.map(natural_key),
).to_csv(
    OUTPUT_ROOT / "learning_legacy_rows_v1_1.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"\n[STEP 7] Learning 레코드 수: {len(learning_legacy)}")
print(f"[STEP 7] 각 bearing 1개씩: {learning_rows_unique}")
print(f"[STEP 7] 발생 횟수: {learning_occurrences}")
display(learning_legacy[learning_output_columns])

Legacy CSV columns:
['bearing', 'N', 'n_reg', 'alarm_idx', 'life_pct', 'lead_frames', 'lead_hours', 'reg_FAR', 'reg_ok', 'status']

[STEP 7] Learning 레코드 수: 6
[STEP 7] 각 bearing 1개씩: True
[STEP 7] 발생 횟수: {'Bearing1_1': 1, 'Bearing1_2': 1, 'Bearing2_1': 1, 'Bearing2_2': 1, 'Bearing3_1': 1, 'Bearing3_2': 1}


,_logical_bearing_id,_N_numeric,_alarm_idx_numeric,_detected,_lead_hours_numeric,_reg_far_numeric_raw,_reg_far_numeric
11,Bearing1_1,2803,1462.0,True,3.7,0.0033,0.0033
12,Bearing1_2,871,826.0,True,0.1,0.0000,0.0000
13,Bearing2_1,911,849.0,True,0.2,0.0088,0.0088
14,Bearing2_2,797,274.0,True,1.5,0.0050,0.0050
15,Bearing3_1,515,496.0,True,0.1,0.0078,0.0078
16,Bearing3_2,1637,1449.0,True,0.5,0.0024,0.0024


In [7]:
# ================================================================
# 8. Legacy CSV 집계와 Frozen JSON 비교
# ================================================================

legacy_metrics = {
    "n_total_learn": int(len(learning_legacy)),
    "n_detected": int(learning_legacy["_detected"].sum()),
    "mean_lead_hours": safe_float(
        learning_legacy.loc[learning_legacy["_detected"], "_lead_hours_numeric"].mean()
    ),
    "max_lead_hours": safe_float(
        learning_legacy.loc[learning_legacy["_detected"], "_lead_hours_numeric"].max()
    ),
    "min_lead_hours": safe_float(
        learning_legacy.loc[learning_legacy["_detected"], "_lead_hours_numeric"].min()
    ),
    "mean_reg_FAR": safe_float(learning_legacy["_reg_far_numeric"].mean()),
    "max_reg_FAR": safe_float(learning_legacy["_reg_far_numeric"].max()),
}

print("[STEP 8] Legacy 집계 지표:")
for k, v in legacy_metrics.items():
    print(f"  {k}: {v}")

print("\n[STEP 8] Frozen JSON 지표:")
for k in METRIC_TOLERANCES:
    print(f"  {k}: {frozen.get(k)}")

consistency_rows = []

for metric_name, tolerance in METRIC_TOLERANCES.items():
    frozen_value = safe_float(frozen.get(metric_name))
    legacy_value = safe_float(legacy_metrics.get(metric_name))

    if frozen_value is None or legacy_value is None:
        abs_difference = None
        passed = False
        status = "MISSING_VALUE"
    else:
        abs_difference = abs(frozen_value - legacy_value)
        passed = abs_difference <= tolerance
        status = "PASS" if passed else "FAIL"

    consistency_rows.append({
        "metric": metric_name,
        "frozen_value": frozen_value,
        "legacy_csv_value": legacy_value,
        "absolute_difference": abs_difference,
        "tolerance": tolerance,
        "status": status,
        "comparison_basis": "Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교",
    })

consistency_df = pd.DataFrame(consistency_rows)
consistency_df.to_csv(
    OUTPUT_ROOT / "frozen_legacy_consistency_v1_1.csv",
    index=False,
    encoding="utf-8-sig",
)

frozen_legacy_consistency_pass = bool(
    len(consistency_df) > 0
    and consistency_df["status"].eq("PASS").all()
)

print("\n[CHECKPOINT] Frozen ↔ Legacy 일관성")
display(consistency_df)
print(f"frozen_legacy_consistency_pass = {frozen_legacy_consistency_pass}")

[STEP 8] Legacy 집계 지표:
  n_total_learn: 6
  n_detected: 6
  mean_lead_hours: 1.0166666666666666
  max_lead_hours: 3.7
  min_lead_hours: 0.1
  mean_reg_FAR: 0.004549999999999999
  max_reg_FAR: 0.0088

[STEP 8] Frozen JSON 지표:
  n_total_learn: 6
  n_detected: 6
  mean_lead_hours: 1.02
  max_lead_hours: 3.7
  min_lead_hours: 0.1
  mean_reg_FAR: 0.0045
  max_reg_FAR: 0.0088

[CHECKPOINT] Frozen ↔ Legacy 일관성


,metric,frozen_value,legacy_csv_value,absolute_difference,tolerance,status,comparison_basis
0,n_total_learn,6.0000,6.000000,0.000000,0.00000,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
1,n_detected,6.0000,6.000000,0.000000,0.00000,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
2,mean_lead_hours,1.0200,1.016667,0.003333,0.01500,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
3,max_lead_hours,3.7000,3.700000,0.000000,0.05500,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
4,min_lead_hours,0.1000,0.100000,0.000000,0.05500,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
5,mean_reg_FAR,0.0045,0.004550,0.000050,0.00055,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
6,max_reg_FAR,0.0088,0.008800,0.000000,0.00055,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교


frozen_legacy_consistency_pass = True


In [8]:
# ================================================================
# 9. Frozen pass criteria 충돌 확인
# ================================================================

pass_criteria = frozen.get("pass_criteria", {}) or {}

criteria_min_detected = safe_float(pass_criteria.get("min_detected"))
criteria_max_reg_far  = safe_float(pass_criteria.get("max_reg_FAR"))

frozen_n_detected  = safe_float(frozen.get("n_detected"))
frozen_max_reg_far = safe_float(frozen.get("max_reg_FAR"))

criterion_detected_pass = (
    criteria_min_detected is not None
    and frozen_n_detected is not None
    and frozen_n_detected >= criteria_min_detected
)

criterion_far_pass = (
    criteria_max_reg_far is not None
    and frozen_max_reg_far is not None
    and frozen_max_reg_far <= criteria_max_reg_far
)

criteria_warning = not (criterion_detected_pass and criterion_far_pass)

criteria_report = {
    "source": str(frozen_json_path),
    "pass_criteria": pass_criteria,
    "frozen_result": {
        "n_detected": frozen_n_detected,
        "max_reg_FAR": frozen_max_reg_far,
    },
    "criterion_evaluation": {
        "min_detected_pass": criterion_detected_pass,
        "max_reg_FAR_pass": criterion_far_pass,
    },
    "criteria_warning": criteria_warning,
    "warning_code":
        "FROZEN_BASELINE_SELF_CRITERIA_CONTRADICTION"
        if criteria_warning else None,
    "policy":
        "기준 JSON을 수정하지 않고 경고로 분리한다. "
        "이 경고는 M0 기준 산출물의 무결성 감사를 실패시키지 않는다.",
    "proposed_criteria_applied": False,
}

write_json(OUTPUT_ROOT / "frozen_criteria_warning_v1_1.json", criteria_report)

print(f"[STEP 9] criteria_warning = {criteria_warning}")
print(f"  min_detected 기준: {criteria_min_detected}, 실제: {frozen_n_detected} → {criterion_detected_pass}")
print(f"  max_reg_FAR 기준: {criteria_max_reg_far}, 실제: {frozen_max_reg_far} → {criterion_far_pass}")
if criteria_warning:
    print("  ⚠️  FROZEN_BASELINE_SELF_CRITERIA_CONTRADICTION — 경고로 기록, 감사 실패 아님")

[STEP 9] criteria_warning = True
  min_detected 기준: 5.0, 실제: 6.0 → True
  max_reg_FAR 기준: 0.0, 실제: 0.0088 → False
  ⚠️  FROZEN_BASELINE_SELF_CRITERIA_CONTRADICTION — 경고로 기록, 감사 실패 아님


In [9]:
# ================================================================
# 10. 기존 v1 재계산 결과의 처리
# ================================================================

previous_v1_files = sorted(
    PROJECT_ROOT.glob("audit_outputs/*/m0_learning_recomputed.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

rejected_v1_report = {
    "status": "REFERENCE_REIMPLEMENTATION_REJECTED",
    "files_found": [str(p) for p in previous_v1_files],
    "used_for_frozen_metric_validation": False,
    "used_for_m0_reference": False,
    "reason": [
        "v1 감사 코드가 기준 노트북 로직을 정확히 재현하지 못함",
        "K=6.0을 거리 임계값처럼 취급했을 가능성이 있음",
        "Learning alarm index 및 reg_FAR가 legacy 결과와 불일치함",
        "잘못된 재구현 결과를 기준으로 계속 감사하면 무한 감사 루프가 발생함",
    ],
    "policy":
        "해당 결과는 감사 실패 증거가 아니라 "
        "감사 재구현을 기준 구현으로 채택할 수 없다는 증거로만 사용",
}

if previous_v1_files:
    rejected_v1_report["latest_file"] = file_metadata(previous_v1_files[0])

write_json(OUTPUT_ROOT / "rejected_v1_recomputation_v1_1.json", rejected_v1_report)

print(f"[STEP 10] v1 재계산 파일 발견: {len(previous_v1_files)}개")
print(f"[STEP 10] 상태: REFERENCE_REIMPLEMENTATION_REJECTED")
print(f"[STEP 10] 감사 기준으로 사용: False")

[STEP 10] v1 재계산 파일 발견: 1개
[STEP 10] 상태: REFERENCE_REIMPLEMENTATION_REJECTED
[STEP 10] 감사 기준으로 사용: False


In [10]:
# ================================================================
# 11. 기존 v1 데이터 감사 결과 계승
# ================================================================

inherit_names = [
    "test_fulltest_prefix_audit.csv",
    "test_fulltest_file_comparison.csv",
    "femto_split_manifest.csv",
    "duplicate_name_audit.csv",
]

inherited_rows = []

for artifact_name in inherit_names:
    candidates = sorted(
        PROJECT_ROOT.glob(f"audit_outputs/*/{artifact_name}"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    # 현재 v1.1 폴더는 검색 결과에서 제외
    candidates = [p for p in candidates if OUTPUT_ROOT not in p.parents]

    if not candidates:
        inherited_rows.append({
            "artifact_name": artifact_name,
            "source_path": None,
            "source_sha256": None,
            "copied_path": None,
            "status": "NOT_FOUND_OPTIONAL",
        })
        continue

    source = candidates[0]
    destination = OUTPUT_ROOT / f"inherited_{artifact_name}"
    shutil.copy2(source, destination)

    h_src = sha256_file(source)
    h_dst = sha256_file(destination)

    inherited_rows.append({
        "artifact_name": artifact_name,
        "source_path": str(source),
        "source_sha256": h_src,
        "copied_path": str(destination),
        "copied_sha256": h_dst,
        "status": "COPIED_HASH_MATCH" if h_src == h_dst else "COPY_HASH_MISMATCH",
    })

inherited_df = pd.DataFrame(inherited_rows)
inherited_df.to_csv(
    OUTPUT_ROOT / "inherited_v1_artifacts_v1_1.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"[STEP 11] v1 아티팩트 계승:")
display(inherited_df[["artifact_name", "status"]])

[STEP 11] v1 아티팩트 계승:


,artifact_name,status
0,test_fulltest_prefix_audit.csv,COPIED_HASH_MATCH
1,test_fulltest_file_comparison.csv,NOT_FOUND_OPTIONAL
2,femto_split_manifest.csv,COPIED_HASH_MATCH
3,duplicate_name_audit.csv,COPIED_HASH_MATCH


In [11]:
# ================================================================
# 12. 감사 후 원본 SHA-256 및 불변성 검사
# ================================================================

source_after = {
    str(path.resolve()): file_metadata(path)
    for path in source_paths
}

immutability_rows = []

for path_text, before_metadata in source_before.items():
    after_metadata = source_after.get(path_text)
    unchanged = bool(
        after_metadata
        and before_metadata["sha256"] == after_metadata["sha256"]
        and before_metadata["file_size"] == after_metadata["file_size"]
    )
    immutability_rows.append({
        "absolute_path": path_text,
        "file_name": before_metadata["file_name"],
        "file_size_before": before_metadata["file_size"],
        "file_size_after": after_metadata["file_size"] if after_metadata else None,
        "sha256_before": before_metadata["sha256"],
        "sha256_after": after_metadata["sha256"] if after_metadata else None,
        "unchanged": unchanged,
        "status": "UNCHANGED" if unchanged else "SOURCE_MODIFIED_OR_MISSING",
    })

immutability_df = pd.DataFrame(immutability_rows)
immutability_df.to_csv(
    OUTPUT_ROOT / "source_file_immutability_check_v1_1.csv",
    index=False,
    encoding="utf-8-sig",
)

source_files_unchanged = bool(
    len(immutability_df) > 0
    and immutability_df["unchanged"].all()
)

print(f"\n[STEP 12] source_files_unchanged = {source_files_unchanged}")
if not source_files_unchanged:
    print("🚨 AUDIT_INVALID_SOURCE_MODIFIED")
else:
    print("✅ 원본 파일 변경 없음")
display(immutability_df[["file_name","sha256_before","sha256_after","status"]])


[STEP 12] source_files_unchanged = True
✅ 원본 파일 변경 없음


,file_name,sha256_before,sha256_after,status
0,BASELINE_M0_frozen.json,a4632ad1b9c519768081cea2c35238598f92144580573d...,a4632ad1b9c519768081cea2c35238598f92144580573d...,UNCHANGED
1,m0_baseline_result.csv,63f493e6e3482157dba7a3b7eef75d6d679086bc2ae5b2...,63f493e6e3482157dba7a3b7eef75d6d679086bc2ae5b2...,UNCHANGED
2,M0_FEMTO_Baseline_v1_baseline고정.ipynb,967c36a127d63eee2909adf18edd91f729e4a6796c4d8d...,967c36a127d63eee2909adf18edd91f729e4a6796c4d8d...,UNCHANGED
3,M0_FEMTO_Baseline_v1.ipynb,921f1bbea7b460d748c6dc3f74c04981f3bf4de5ff7622...,921f1bbea7b460d748c6dc3f74c04981f3bf4de5ff7622...,UNCHANGED
4,OnlineSim_FEMTO_v3.ipynb,ce48b07d70af97aa9f9a7c54858aeedd4041e63dd19d00...,ce48b07d70af97aa9f9a7c54858aeedd4041e63dd19d00...,UNCHANGED


In [12]:
# ================================================================
# 13. 최종 판정
# ================================================================

learning_identity_pass = bool(
    len(learning_legacy) == 6
    and learning_rows_unique
    and set(learning_legacy["_logical_bearing_id"]) == set(LEARNING_BEARINGS)
)

required_gate = {
    "dataset_structure_6_11_11_28": dataset_structure_pass,
    "learning_six_records_identified": learning_identity_pass,
    "frozen_legacy_consistency": frozen_legacy_consistency_pass,
    "reference_notebook_checked": bool(notebook_checked),
    "source_files_unchanged": source_files_unchanged,
}

required_gate_pass = all(required_gate.values())

if not source_files_unchanged:
    final_status = "AUDIT_INVALID_SOURCE_MODIFIED"
    next_action = "MANUAL_REVIEW_REQUIRED"
elif required_gate_pass and criteria_warning:
    final_status = "AUDIT_PASS_WITH_CRITERIA_WARNING"
    next_action = "READY_FOR_M0_BRIDGE_2048"
elif required_gate_pass:
    final_status = "AUDIT_PASS"
    next_action = "READY_FOR_M0_BRIDGE_2048"
elif not dataset_structure_pass:
    final_status = "AUDIT_HOLD"
    next_action = "FIX_DATA_IDENTITY_FIRST"
elif not learning_identity_pass:
    final_status = "AUDIT_HOLD"
    next_action = "FIX_REFERENCE_MAPPING_FIRST"
elif not frozen_legacy_consistency_pass:
    final_status = "AUDIT_HOLD"
    next_action = "MANUAL_REVIEW_REQUIRED"
else:
    final_status = "AUDIT_HOLD"
    next_action = "MANUAL_REVIEW_REQUIRED"

print("=" * 72)
print("Required Gate:")
for gate, result in required_gate.items():
    icon = "✅" if result else "❌"
    print(f"  {icon} {gate}: {result}")
print(f"  required_gate_pass = {required_gate_pass}")
print(f"  criteria_warning   = {criteria_warning}")
print("=" * 72)
print(f"FINAL STATUS : {final_status}")
print(f"NEXT ACTION  : {next_action}")
print("=" * 72)

Required Gate:
  ✅ dataset_structure_6_11_11_28: True
  ✅ learning_six_records_identified: True
  ✅ frozen_legacy_consistency: True
  ✅ reference_notebook_checked: True
  ✅ source_files_unchanged: True
  required_gate_pass = True
  criteria_warning   = True
FINAL STATUS : AUDIT_PASS_WITH_CRITERIA_WARNING
NEXT ACTION  : READY_FOR_M0_BRIDGE_2048


In [13]:
# ================================================================
# 14. 최종 보고서 생성 (JSON + Markdown)
# ================================================================

final_summary = {
    "audit_version": AUDIT_VERSION,
    "created_at": now_iso(),
    "final_status": final_status,
    "recommended_next_action": next_action,
    "required_gate": required_gate,
    "required_gate_pass": required_gate_pass,
    "criteria_warning": criteria_warning,
    "criteria_warning_code": criteria_report.get("warning_code"),
    "project_root": str(PROJECT_ROOT),
    "femto_root": str(FEMTO_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "selected_reference_files": {
        "frozen_json": file_metadata(frozen_json_path),
        "legacy_result_csv": file_metadata(legacy_csv_path),
        "reference_notebook": file_metadata(reference_notebook_path),
    },
    "dataset_counts": split_count_summary,
    "dataset_count_checks": dataset_count_checks,
    "learning_record_count": len(learning_legacy),
    "learning_occurrences": learning_occurrences,
    "legacy_metrics": legacy_metrics,
    "frozen_metrics": {name: frozen.get(name) for name in METRIC_TOLERANCES},
    "algorithm_reimplementation": {
        "performed": False,
        "v1_recomputation_status": "REFERENCE_REIMPLEMENTATION_REJECTED",
        "reason": "기준 노트북의 실제 구현을 추정하여 다시 작성하지 않음",
    },
    "alarm_timing_contract": {
        "snapshot_duration_sec": FEMTO_SNAPSHOT_DURATION_SEC,
        "decision_interval_sec": FEMTO_DECISION_INTERVAL_SEC,
        "CONSEC": CONSEC,
        "decision_span_sec": FEMTO_DECISION_INTERVAL_SEC * CONSEC,
        "start_to_confirm_difference_sec": FEMTO_DECISION_INTERVAL_SEC * (CONSEC - 1),
    },
    "source_files_unchanged": source_files_unchanged,
}

write_json(OUTPUT_ROOT / "FINAL_AUDIT_SUMMARY_v1_1.json", final_summary)


def bool_mark(value):
    return "PASS" if value else "FAIL"


try:
    consistency_markdown = consistency_df.to_markdown(index=False)
except Exception:
    consistency_markdown = consistency_df.to_string(index=False)

try:
    immutability_markdown = immutability_df[["file_name","sha256_before","sha256_after","status"]].to_markdown(index=False)
except Exception:
    immutability_markdown = immutability_df[["file_name","status"]].to_string(index=False)

learning_display = learning_legacy[learning_output_columns].sort_values(
    "_logical_bearing_id", key=lambda s: s.map(natural_key)
)
try:
    learning_markdown = learning_display.to_markdown(index=False)
except Exception:
    learning_markdown = learning_display.to_string(index=False)

summary_md = f"""# M0_AUDIT_v1.1 Final Audit Summary

## 1. Final decision

- **Audit status:** `{final_status}`
- **Recommended next action:** `{next_action}`
- **Required gate pass:** `{required_gate_pass}`
- **Criteria warning:** `{criteria_warning}`
- **Audit timestamp:** `{now_iso()}`

## 2. Audit policy

이번 v1.1에서는 M0 알고리즘을 감사 코드에서 임의로 재구현하지 않았다.

기존 v1의 재계산 결과는 다음 상태로 분류한다.

- `REFERENCE_REIMPLEMENTATION_REJECTED`
- Frozen 기준 검증에 사용하지 않음
- M0 기준 구현으로 사용하지 않음
- 전체 감사 실패 사유로 사용하지 않음

검증 기준은 다음 세 가지 동결 자료이다.

1. `BASELINE_M0_frozen.json`
2. `m0_baseline_result.csv`
3. `{reference_notebook_path.name}` 및 그 SHA-256

## 3. Required gate

| Gate | Result |
|---|---:|
| Dataset structure 6/11/11/28 | {bool_mark(dataset_structure_pass)} |
| Learning 6 records identified | {bool_mark(learning_identity_pass)} |
| Frozen ↔ legacy consistency | {bool_mark(frozen_legacy_consistency_pass)} |
| Reference notebook checked | {bool_mark(bool(notebook_checked))} |
| Source files unchanged | {bool_mark(source_files_unchanged)} |

## 4. Dataset structure

- LEARNING: {split_count_summary.get("LEARNING")}
- TEST: {split_count_summary.get("TEST")}
- FULL_TEST: {split_count_summary.get("FULL_TEST")}
- TOTAL: {split_count_summary.get("TOTAL")}

## 5. Learning legacy records

{learning_markdown}

## 6. Frozen ↔ legacy consistency

{consistency_markdown}

## 7. Frozen pass-criteria warning

Frozen JSON에 저장된 조건:

- `min_detected = {criteria_min_detected}`
- `max_reg_FAR = {criteria_max_reg_far}`

Frozen 기준 결과:

- `n_detected = {frozen_n_detected}`
- `max_reg_FAR = {frozen_max_reg_far}`

평가:

- Detected criterion: `{bool_mark(criterion_detected_pass)}`
- FAR criterion: `{bool_mark(criterion_far_pass)}`

따라서 이 충돌은
`FROZEN_BASELINE_SELF_CRITERIA_CONTRADICTION`으로 기록한다.

Frozen JSON은 변경하지 않았으며 새로운 기준도 적용하지 않았다.
이 항목은 기준 파일의 무결성 감사 실패가 아니라
**기준 정의상의 경고**이다.

## 8. Alarm timing contract

- Snapshot duration: {FEMTO_SNAPSHOT_DURATION_SEC} sec
- Decision interval: {FEMTO_DECISION_INTERVAL_SEC} sec
- CONSEC: {CONSEC}
- Five-decision span: {FEMTO_DECISION_INTERVAL_SEC * CONSEC} sec
- Alarm start-to-confirm difference: {FEMTO_DECISION_INTERVAL_SEC * (CONSEC - 1)} sec

## 9. Source immutability

{immutability_markdown}

## 10. Reference notebook

- Selected file: `{reference_notebook_path}`
- SHA-256: `{sha256_file(reference_notebook_path)}`
- Code-cell count: `{len(code_cells)}`
- Inspection policy: `READ_ONLY_STATIC_INSPECTION`
- Algorithm execution: `False`
- Algorithm reimplementation: `False`

## 11. Audit closure rule

데이터 구조, Learning 기준 레코드, frozen/legacy 일관성,
기준 노트북 확인 및 원본 불변성이 모두 통과하면
M0 감사는 종료한다.

Frozen pass criteria의 자기모순은 경고로 유지하되,
M0-BRIDGE-2048 진행을 차단하지 않는다.

## 12. Final conclusion

- **Final status:** `{final_status}`
- **Next action:** `{next_action}`
"""

with (OUTPUT_ROOT / "FINAL_AUDIT_SUMMARY_v1_1.md").open("w", encoding="utf-8") as f:
    f.write(summary_md)

print(f"[STEP 14] FINAL_AUDIT_SUMMARY_v1_1.md 저장 완료")
print(f"[STEP 14] FINAL_AUDIT_SUMMARY_v1_1.json 저장 완료")

[STEP 14] FINAL_AUDIT_SUMMARY_v1_1.md 저장 완료
[STEP 14] FINAL_AUDIT_SUMMARY_v1_1.json 저장 완료


In [14]:
# ================================================================
# 15. 최종 필수 검증
# ================================================================

required_output_names = [
    "project_resolution_v1_1.json",
    "source_code_contract_v1_1.json",
    "dataset_structure_v1_1.csv",
    "learning_legacy_rows_v1_1.csv",
    "frozen_legacy_consistency_v1_1.csv",
    "frozen_criteria_warning_v1_1.json",
    "rejected_v1_recomputation_v1_1.json",
    "inherited_v1_artifacts_v1_1.csv",
    "source_file_immutability_check_v1_1.csv",
    "FINAL_AUDIT_SUMMARY_v1_1.md",
    "FINAL_AUDIT_SUMMARY_v1_1.json",
]

missing_outputs = [
    name for name in required_output_names
    if not (OUTPUT_ROOT / name).exists()
]

if missing_outputs:
    raise RuntimeError(
        f"필수 출력 파일이 누락되었습니다: {missing_outputs}"
    )

# 파라미터 assert 재확인
assert CONSEC == 5
assert FEMTO_DECISION_INTERVAL_SEC == 10.0
assert FEMTO_SNAPSHOT_DURATION_SEC == 0.1
assert COL_H == 4
assert K == 6.0
assert REG_MIN == 90
assert REG_MAX == 600

# 원본 변경은 어떤 경우에도 허용하지 않음
if not source_files_unchanged:
    raise RuntimeError(
        "AUDIT_INVALID_SOURCE_MODIFIED: 원본 파일 해시가 변경되었습니다."
    )

print("\n" + "=" * 72)
print("M0_AUDIT_v1.1 완료")
print(f"FINAL STATUS : {final_status}")
print(f"NEXT ACTION  : {next_action}")
print(f"OUTPUT ROOT  : {OUTPUT_ROOT}")
print("=" * 72)

print("\n[Frozen ↔ Legacy 일관성]")
display(consistency_df)

print("\n[원본 불변성]")
display(immutability_df[["file_name","status"]])

print("\n생성 파일:")
for path in sorted(OUTPUT_ROOT.iterdir()):
    print("-", path.name)


M0_AUDIT_v1.1 완료
FINAL STATUS : AUDIT_PASS_WITH_CRITERIA_WARNING
NEXT ACTION  : READY_FOR_M0_BRIDGE_2048
OUTPUT ROOT  : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_055925_v1_1

[Frozen ↔ Legacy 일관성]


,metric,frozen_value,legacy_csv_value,absolute_difference,tolerance,status,comparison_basis
0,n_total_learn,6.0000,6.000000,0.000000,0.00000,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
1,n_detected,6.0000,6.000000,0.000000,0.00000,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
2,mean_lead_hours,1.0200,1.016667,0.003333,0.01500,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
3,max_lead_hours,3.7000,3.700000,0.000000,0.05500,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
4,min_lead_hours,0.1000,0.100000,0.000000,0.05500,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
5,mean_reg_FAR,0.0045,0.004550,0.000050,0.00055,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교
6,max_reg_FAR,0.0088,0.008800,0.000000,0.00055,PASS,Frozen JSON 표시 정밀도를 고려한 legacy CSV 집계 비교



[원본 불변성]


,file_name,status
0,BASELINE_M0_frozen.json,UNCHANGED
1,m0_baseline_result.csv,UNCHANGED
2,M0_FEMTO_Baseline_v1_baseline고정.ipynb,UNCHANGED
3,M0_FEMTO_Baseline_v1.ipynb,UNCHANGED
4,OnlineSim_FEMTO_v3.ipynb,UNCHANGED



생성 파일:
- FINAL_AUDIT_SUMMARY_v1_1.json
- FINAL_AUDIT_SUMMARY_v1_1.md
- dataset_structure_v1_1.csv
- frozen_criteria_warning_v1_1.json
- frozen_legacy_consistency_v1_1.csv
- inherited_duplicate_name_audit.csv
- inherited_femto_split_manifest.csv
- inherited_test_fulltest_prefix_audit.csv
- inherited_v1_artifacts_v1_1.csv
- learning_legacy_rows_v1_1.csv
- project_resolution_v1_1.json
- rejected_v1_recomputation_v1_1.json
- source_code_contract_v1_1.json
- source_file_immutability_check_v1_1.csv
